# Summarize and plot felsenstein codes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
filename = "chr12_HapMapII_GRCh37_CEU50_seed1"
percent = "10"
num_bins = "20"

df = pd.read_csv(f"../data/felsenstein_{filename}_{percent}.00percent_{num_bins}bins_seed42.csv")

# ---------------------------------------------------------------
# 1. headline: true tree vs Relate, side by side
#    joint = the single best history; marginal = all histories added up
# ---------------------------------------------------------------
summary = pd.DataFrame({
    name: {
        "SNPs": len(df),
        # best (joint) history
        "BEST: total log probability": df[f"{name}_log_joint"].sum(),
        "BEST: mean log probability per SNP": df[f"{name}_log_joint"].mean(),
        "BEST: mean mutations per SNP": df[f"{name}_mutations"].mean(),
        "BEST: % SNPs explained by 1 mutation": 100 * (df[f"{name}_mutations"] == 1).mean(),
        "BEST: % SNPs with a back mutation": 100 * (df[f"{name}_back_mutations"] > 0).mean(),
        "BEST: % SNPs with >1 new mutation": 100 * (df[f"{name}_new_mutations"] > 1).mean(),
        "BEST: % true time inside chosen branch": 100 * df[f"{name}_time_inside"].mean(),
        "BEST: median share of the marginal": df[f"{name}_best_share"].median(),
        "BEST: % SNPs where best history < 90% of marginal": 100 * (df[f"{name}_best_share"] < 0.9).mean(),
        # all histories (marginal)
        "MARGINAL: total log likelihood": df[f"{name}_log_marginal"].sum(),
        "MARGINAL: mean log likelihood per SNP": df[f"{name}_log_marginal"].mean(),
        # clade correlation
        "CLADE: % best clade correlation = 1": 100 * np.isclose(df[f"{name}_best_clade_corr"], 1).mean(),
        "CLADE: median best clade correlation": df[f"{name}_best_clade_corr"].median(),
    }
    for name in ("std", "relate")
})
print(summary.round(3).to_string())


In [ ]:

# ---------------------------------------------------------------
# 2. per SNP: how much worse is Relate than the truth?
# ---------------------------------------------------------------
gaps = pd.DataFrame({
    "best (joint)": df.dlog_best,
    "marginal": df.dlog_marginal,
})
print("\nlog score, Relate minus truth (0 = equally good, negative = Relate worse)")
print(gaps.describe().round(2).to_string())
for col in gaps:
    print(f"  {col:13s} Relate as good as or better than truth: {100 * (gaps[col] >= -1e-6).mean():.1f}% of SNPs")
print(f"same number of mutations in the best history as on the true tree: "
      f"{100 * (df.relate_mutations == df.std_mutations).mean():.1f}% of SNPs")


In [ ]:

# ---------------------------------------------------------------
# 3. how Relate's best history explains each SNP
# ---------------------------------------------------------------
perfect = (df.relate_mutations == 1)
extra_inside = (df.relate_new_mutations == 1) & (df.relate_back_mutations > 0)
split = (df.relate_new_mutations > 1) & (df.relate_back_mutations == 0)
both = (df.relate_new_mutations > 1) & (df.relate_back_mutations > 0)
outcome = pd.Series({
    "1 mutation (carriers form one clade)": perfect.mean(),
    "1 new + back mutation(s) (non-carriers inside the clade)": extra_inside.mean(),
    ">1 new mutations (carriers split across clades)": split.mean(),
    "both": both.mean(),
}) * 100
print("\nhow Relate's best history explains the hidden SNPs (% of SNPs):")
print(outcome.round(1).to_string())


In [ ]:

# ---------------------------------------------------------------
# 4. by ALT count bin
# ---------------------------------------------------------------
by_freq = df.groupby("ALT_count_bin").agg(
    ALT_count_range=("ALT_count", lambda x: f"{x.min()}-{x.max()}"),
    SNPs=("position", "size"),
    pct_relate_1_mutation=("relate_mutations", lambda x: 100 * (x == 1).mean()),
    mean_relate_mutations=("relate_mutations", "mean"),
    median_dlog_best=("dlog_best", "median"),
    median_dlog_marginal=("dlog_marginal", "median"),
    median_relate_best_share=("relate_best_share", "median"),
    median_relate_best_clade_corr=("relate_best_clade_corr", "median"),
    pct_relate_time_inside=("relate_time_inside", lambda x: 100 * x.mean()),
)
print("\nby ALT count bin:")
print(by_freq.round(2).to_string())


In [ ]:

# ---------------------------------------------------------------
# 5. the SNPs where Relate's best history does worst
# ---------------------------------------------------------------
print("\n10 SNPs where Relate's best history does worst:")
print(df.nsmallest(10, "dlog_best")[
    ["position", "ALT_count", "std_mutations", "relate_mutations", "relate_new_mutations",
     "relate_back_mutations", "relate_best_share", "relate_best_clade_corr", "dlog_best", "dlog_marginal"]
].round(3).to_string(index=False))

In [ ]:
multi = df[df.true_mutations > 1]
print(f"{len(multi)} of {len(df)} usable hidden SNPs have more than one true mutation")
print(df.true_mutations.value_counts().sort_index())

In [ ]:
multi = df[df.true_mutations > 1].sort_values("ALT_count")
print(f"{len(multi)} SNPs with more than one true mutation\n")
print(multi[["position", "ALT_count", "ALT_count_bin", "true_mutations",
             "std_mutations", "relate_mutations"]].to_string(index=False))

In [ ]:
print(multi.ALT_count_bin.value_counts().sort_index())

# PLOTS

In [ ]:
# colours: same palette as plot_tree.py; blue = true tree, orange = Relate
SURFACE, INK, INK_2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
TRUE_C, RELATE_C = "#2a78d6", "#eb6834"
OUTCOME_C = ["#1baf7a", "#eda100", "#e87ba4", "#6250d6"]          # aqua, yellow, magenta, violet (blue/orange already mean true/Relate)
OUTCOME_TEXT = [INK, INK, INK, SURFACE]                             # label ink that reads on each fill

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "legend.frameon": False, "legend.fontsize": 9,
})

# ALT count bins: use the bins already in results_df, labelled by the ALT counts actually in each
present = sorted(df.ALT_count_bin.dropna().unique())
ranges = df.groupby("ALT_count_bin").ALT_count.agg(["min", "max"])
bin_labels = [f"{ranges.loc[b, 'min']}-{ranges.loc[b, 'max']}" for b in present]
x = np.arange(len(present))

# ---------------------------------------------------------------
# 1. how much less likely does Relate make each hidden SNP?
# ---------------------------------------------------------------
orders = -df.dlog_marginal / np.log(10)           # orders of magnitude less likely than on the true tree
gap_bins = [-np.inf, 0, 1, 2, 5, 10, 50, np.inf]
gap_labels = ["as likely\nor more", "<10×", "10–100×", "10²–10⁵×", "10⁵–10¹⁰×", "10¹⁰–10⁵⁰×", ">10⁵⁰×"]
counts = pd.cut(orders, bins=gap_bins, labels=gap_labels).value_counts().reindex(gap_labels)
pct = 100 * counts / len(df)

fig, ax = plt.subplots(figsize=(8, 3.8))
bars = ax.bar(gap_labels, pct, color=RELATE_C, width=0.7, edgecolor=SURFACE, linewidth=2)
for b, p in zip(bars, pct):
    ax.text(b.get_x() + b.get_width() / 2, p + 0.8, f"{p:.0f}%", ha="center", va="bottom",
            fontsize=9, color=INK_2)
ax.set_ylabel("% of hidden SNPs")
ax.set_title("How much less likely Relate's tree makes each hidden SNP", pad=24)
ax.text(0, 1.03, "marginal likelihood on the true tree ÷ on the Relate tree",
        transform=ax.transAxes, fontsize=9, color=MUTED, va="bottom")
ax.set_ylim(0, pct.max() * 1.15)
ax.tick_params(axis="x", length=0)
fig.tight_layout()
plt.show()

# ---------------------------------------------------------------
# 2. by ALT count bin: true tree vs Relate, two measures
# ---------------------------------------------------------------
g = df.groupby("ALT_count_bin")
panels = [
    ("Explained by a single mutation", lambda d, n: 100 * (d[f"{n}_mutations"] == 1).mean()),
    ("True mutation time inside the chosen branch", lambda d, n: 100 * d[f"{n}_time_inside"].mean()),
]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharey=True)
for ax, (title, stat) in zip(axes, panels):
    for name, colour, label in (("std", TRUE_C, "true tree"), ("relate", RELATE_C, "Relate tree")):
        y = [stat(g.get_group(b), name) for b in present]
        ax.plot(x, y, color=colour, linewidth=2, marker="o", markersize=7,
                markeredgecolor=SURFACE, markeredgewidth=2, label=label)
        ax.text(x[-1] + 0.3, y[-1], label, color=INK_2, fontsize=9, va="center")   # direct label
    ax.set_xticks(x, bin_labels, rotation=60, ha="right")
    ax.set_xlim(-0.5, len(present) - 0.5 + 2.5)
    ax.set_ylim(-3, 105)
    ax.set_xlabel("ALT count")
    ax.set_title(title)
    ax.tick_params(length=0)
axes[0].set_ylabel("% of hidden SNPs")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", ncol=2)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

# ---------------------------------------------------------------
# 3. how Relate explains the hidden SNPs, by ALT count bin
# ---------------------------------------------------------------
kind = np.select(
    [df.relate_mutations == 1,
     (df.relate_new_mutations == 1) & (df.relate_back_mutations > 0),
     (df.relate_new_mutations > 1) & (df.relate_back_mutations == 0)],
    ["one mutation", "non-carriers inside the clade (back mutations)",
     "carriers split across clades (extra new mutations)"],
    default="both")
order = ["one mutation", "non-carriers inside the clade (back mutations)",
         "carriers split across clades (extra new mutations)", "both"]
share = (pd.crosstab(df.ALT_count_bin, kind, normalize="index") * 100).reindex(index=present, columns=order).fillna(0)

fig, ax = plt.subplots(figsize=(12, 4.4))
bottom = np.zeros(len(present))
for col, colour, ink in zip(order, OUTCOME_C, OUTCOME_TEXT):
    ax.bar(x, share[col], bottom=bottom, color=colour, width=0.7,
           edgecolor=SURFACE, linewidth=2, label=col)
    for xi, (b, h) in enumerate(zip(bottom, share[col])):
        if h >= 10:                                             # label segments big enough to read
            ax.text(xi, b + h / 2, f"{h:.0f}", ha="center", va="center", fontsize=7, color=ink)
    bottom += share[col].to_numpy()
ax.set_xticks(x, bin_labels, rotation=60, ha="right")
ax.set_xlabel("ALT count")
ax.set_ylabel("% of hidden SNPs")
ax.set_ylim(0, 100)
ax.set_title("How Relate's tree explains each hidden SNP")
ax.tick_params(length=0)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
fig.tight_layout()
plt.show()